# Problem 049:  Prime Permutations

> The arithmetic sequence, $1487, 4817, 8147$, in which each of the terms increases by $3330$, is unusual in two ways: (i) each of the three terms are prime, and, (ii) each of the $4$-digit numbers are permutations of one another.
>
> There are no arithmetic sequences made up of three $1$-, $2$-, or $3$-digit primes, exhibiting this property, but there is one other $4$-digit increasing sequence.
>
> What $12$-digit number do you form by concatenating the three terms in this sequence?

## Solution $\mathcal O\left(10^{3N} /N^{21}\right)$

The goal is to find a sequence of numbers that have three different properties:  they are all prime, they are all anagrams, it is an arithmetic sequence.  The order the code below considers these three properties is to find a prime, find all prime anagrams, and then to determine if there is an arithmetic sequence among them.

For prime generation, the sieve is used.  In fact, the problem is efficiently answered just using the sieve itself, and not the list of prime numbers.  For this problem, I will say that numbers cannot have leading $0$'s, so the sieve is flipped to `False` for all numbers with less digits than desired.

Next, the list of anagrams for a prime are generated using `itertools.permutations` for each prime.  Whenever a prime is found in this process, the sieve is flipped to `False` to avoid double counting sets of primes, and also avoid adding a prime multiple times to a list if there is a repeated digit.

With a list of prime anagrams, arithmetic seuqences are searched for if there are enough to make the desired sequence length.  This is done by looking at each pair of primes in the list, presume they are the first two members of the arithmetic sequence, and seeing if the subsequent numbers are also in the list of anagrams.  This part of the code could be optimized a bit, since a binary search could be implemented instead of the `in` operation.

I am uncertain of the time scaling here, other than to say "bad".  The way I think about it is in a probabilistic sense.  For a set of $N$-digit numbers, there are roughly $10^N$ values.  Out of those numbers, there are ${N+9} \choose {N}$ unique sets of anagrams, each with an average length of $10^N / {{N+9} \choose {N}} \sim 10^N / N^9$.  The probability of any of these numbers being prime is approximately $1/\ln 10^N$, so the average length of set of prime anagrams goes as $10^N / N^{10}$.  The code does an order $M^3$ operation for a given set of prime anagrams of list $M$, first to look at all pairs ($\mathcal O(M^2)$) and then the `in` operation ($\mathcal O(M)$).  Presuming the average cube goes as the cube of the average, the time scaling for one set of prime anagrams is $\mathcal O\left( 10^{3N} / N^{30}\right)$.  The overall time scaling is then this scaling times the number of prime anagram sets, giving $\mathcal O\left( 10^{3N} / N^{21}\right)$.

In [34]:
%%time

from itertools import permutations

def sieve_of_eratosthenes(num: int) -> list[int]:
    sieve = [True] * ((num-1) // 2)
    pmax = int(num**0.5)//2
    for p in range(pmax):
        if sieve[p]:
            k = 2 * p + 3
            l = ((num - 1) // 2 - k**2 // 2) // k + 1
            sieve[k**2//2-1::k] = [False] * l
    #return [2] + [2*i+3 for i in range((num-1)//2) if sieve[i]]
    return sieve

def p049(N: int, M: int) -> list[str]:
    val = []
    s = sieve_of_eratosthenes(pow(10,N))
    
    s[:pow(10,N-1)//2] = [False] * (pow(10,N-1)//2)
    
    for n in range(pow(10, N-1) + 1, pow(10,N), 2):
        if s[n//2-1]:
            plist = []
            for x in permutations(str(n)):
                m = int("".join(x))
                if m % 2 and s[m//2-1]:
                    plist.append(m)
                    s[m//2-1] = False
            if len(plist) >= M:
                for x in combinations(sorted(plist), 2):
                    dx = x[1] - x[0]
                    xp = x[1] + dx
                    for i in range(2, M):
                        if xp not in plist:
                            break
                        xp += dx
                    else:
                        val.append("".join(map(str, [x[0] + i * dx for i in range(M)])))
    return val

N = 4
M = 3
ans = p049(N, M)

print(ans)

['148748178147', '296962999629']
CPU times: user 8.37 ms, sys: 7 μs, total: 8.38 ms
Wall time: 7.89 ms
